# Sementic Chunking

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader,DirectoryLoader
from langchain_experimental.text_splitter import SemanticChunker
from dotenv import load_dotenv
import os

load_dotenv()

True

In [6]:
def load_documents(doc_path):
    """Load documents from the specified path."""

    if not os.path.exists(doc_path):
        raise FileNotFoundError(f"Document not found: {doc_path}")
    
    loader = DirectoryLoader(doc_path, 
                             glob="*.txt",
                             show_progress=True,
                             loader_cls=TextLoader,
                             loader_kwargs={"encoding": "utf-8"})
    
    documents = loader.load()

    if len(documents) == 0:
        raise ValueError(f"No .txt files found in the specified path: {doc_path}")
    
    return documents

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    show_progress=True
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [8]:
semantic_splitter = SemanticChunker(
    embeddings=embedding_model,
    breakpoint_threshold_type="percentile", #or standard_deviation
    breakpoint_threshold_amount=70
)


In [9]:
# Same Tesla text but structured to show semantic grouping
tesla_text = """Tesla's Q3 Results
Tesla reported record revenue of $25.2B in Q3 2024.
The company exceeded analyst expectations by 15%.
Revenue growth was driven by strong vehicle deliveries.

Model Y Performance  
The Model Y became the best-selling vehicle globally, with 350,000 units sold.
Customer satisfaction ratings reached an all-time high of 96%.
Model Y now represents 60% of Tesla's total vehicle sales.

Production Challenges
Supply chain issues caused a 12% increase in production costs.
Tesla is working to diversify its supplier base.
New manufacturing techniques are being implemented to reduce costs."""


chunks = semantic_splitter.split_text(tesla_text)

print("SEMANTIC CHUNKING RESULTS:")
print("=" * 50)
for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}: ({len(chunk)} chars)")
    print(f'"{chunk}"')
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

SEMANTIC CHUNKING RESULTS:
Chunk 1: (120 chars)
"Tesla's Q3 Results
Tesla reported record revenue of $25.2B in Q3 2024. The company exceeded analyst expectations by 15%."

Chunk 2: (219 chars)
"Revenue growth was driven by strong vehicle deliveries. Model Y Performance  
The Model Y became the best-selling vehicle globally, with 350,000 units sold. Customer satisfaction ratings reached an all-time high of 96%."

Chunk 3: (192 chars)
"Model Y now represents 60% of Tesla's total vehicle sales. Production Challenges
Supply chain issues caused a 12% increase in production costs. Tesla is working to diversify its supplier base."

Chunk 4: (67 chars)
"New manufacturing techniques are being implemented to reduce costs."

